In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/huboyang/lab1-shared-new/lab1_shared.py


In [2]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 40.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [3]:
# Install bitsandbytes
%pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.9 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="unsloth/Qwen2.5-3B-Instruct-bnb-4bit")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.05GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'I am Qwen, a large language model created by Alibaba Cloud. I am designed to assist with a variety of tasks including answering questions, providing information, and generating text. How can I assist you today?'}]}]

In [5]:
# Check the environment
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("No GPU detected. Please enable GPU in Notebook settings.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [14]:
import sys
sys.path.insert(0, "/kaggle/input/datasets/huboyang/lab1-shared-new")

import lab1_shared as shared
print(shared.MODEL_NAME)
print(shared.SUPER2HYPER)

unsloth/Qwen2.5-3B-Instruct-bnb-4bit
{'artifact': 'inanimate', 'body': 'inanimate', 'food': 'inanimate', 'object': 'inanimate', 'plant': 'inanimate', 'substance': 'inanimate', 'act': 'dynamic_situation', 'event': 'dynamic_situation', 'phenomenon': 'dynamic_situation', 'animal': 'animate', 'person': 'animate', 'groupxperson': 'animate', 'cognition': 'information', 'communication': 'information', 'quantity': 'quantity', 'attribute': 'stative_situation', 'feeling': 'stative_situation', 'relation': 'stative_situation', 'state': 'stative_situation', 'time': 'time', 'possession': 'possession', 'institution': 'institution', 'act*cognition': 'dynamic_situation*information', 'artifact*cognition': 'inanimate*information', 'other': 'other'}


In [15]:
# Download and load the dataset
import pandas as pd
import numpy as np
import random

shared.set_seed(1)

df = shared.load_data()
splits, hypersense_labels = shared.prepare_data(df)

train_df = splits["train"]
rand_dev = splits["rand-dev"]
rand_test = splits["rand-test"]
freq_test = splits["freq-test"]

print("Labels:", hypersense_labels)

for name, split in splits.items():
    print(f"{name}: {len(split)}")

Labels: ['animate', 'dynamic_situation', 'dynamic_situation*information', 'inanimate', 'inanimate*information', 'information', 'institution', 'other', 'possession', 'quantity', 'stative_situation', 'time']
train: 10592
rand-dev: 570
rand-test: 678
freq-test: 1517


In [16]:
# Majority baseline
from sklearn.metrics import accuracy_score, f1_score

# Predict the most frequent hypersense in the training set
majority_label = train_df["hypersense"].value_counts().idxmax()

print("Majority label:", majority_label)

# Evaluate on rand-dev, rand-test, and freq-test
baseline_results = []

for name, split in {
    "rand-dev": rand_dev,
    "rand-test": rand_test,
    "freq-test": freq_test,
}.items():
    y_true = split["hypersense"]
    y_pred = [majority_label] * len(split)

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=hypersense_labels,
        average="macro",
        zero_division=0,
    )

    baseline_results.append({
        "split": name,
        "majority_label": majority_label,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
    })

baseline_df = pd.DataFrame(baseline_results)

print("\nBaseline results:")
display(baseline_df)


Majority label: inanimate

Baseline results:


,split,majority_label,accuracy,macro_f1
0,rand-dev,inanimate,0.238596,0.032106
1,rand-test,inanimate,0.315634,0.039985
2,freq-test,inanimate,0.208965,0.028808


In [17]:
# Select one training example for each label
few_shot_examples = (
    train_df.groupby("hypersense", group_keys=False)
    .sample(n=1, random_state=1)
    .sort_values("hypersense")
    .reset_index(drop=True)
)

print("Number of few-shot examples:", len(few_shot_examples))
display(few_shot_examples[["lemma", "definition", "hypersense"]])

Number of few-shot examples: 12


,lemma,definition,hypersense
0,Gendreyenne,"Habitante de Gendrey, commune française située...",animate
1,effort,"Pression exercée par un corps, en parlant des ...",dynamic_situation
2,compromis,Acte qui scelle un échange de promesses. C’est...,dynamic_situation*information
3,roussette,"Roussette-de-savoie, vin blanc d’appellation d...",inanimate
4,expertise,"Procès-verbal, rapport des experts.",inanimate*information
5,continuation,Suite d’instructions qui reste à être exécutée...,information
6,pinacothèque,Cabinet de peinture.,institution
7,décharge,"Lieu où l’on entrepose les déchets, les débris.",other
8,ressource,Ce dont on a besoin ; moyen d’action.,possession
9,couple,"Deux forces égales, parallèles et de direction...",quantity


In [18]:
# Build prompts
def build_messages(
    row,
    prompt_language="en",
    use_lemma=True,
    few_shot=False,
):
    labels_text = ", ".join(hypersense_labels)

    if prompt_language == "en":
        system_prompt = (
            "You are a lexical semantic classification system. "
            "Classify each French word definition into exactly one "
            "hypersense label from this list: "
            f"{labels_text}. "
            "The labels must remain in English. "
            "Return only one label, with no explanation."
        )
        instruction = "Classify this entry:"
    else:
        system_prompt = (
            "Vous êtes un système de classification sémantique lexicale. "
            "Classez chaque définition de mot français dans une seule "
            "catégorie parmi les suivantes : "
            f"{labels_text}. "
            "Les étiquettes doivent rester en anglais. "
            "Répondez uniquement par une étiquette, sans explication."
        )
        instruction = "Classez cette entrée :"

    messages = [{"role": "system", "content": system_prompt}]

    if few_shot:
        for _, example in few_shot_examples.iterrows():
            example_text = (
                f"{instruction}\n"
                f"{shared.format_input(example, prompt_language, use_lemma)}"
            )
            messages.append({
                "role": "user",
                "content": example_text,
            })
            messages.append({
                "role": "assistant",
                "content": example["hypersense"],
            })

    target_text = (
        f"{instruction}\n"
        f"{shared.format_input(row, prompt_language, use_lemma)}"
    )
    messages.append({"role": "user", "content": target_text,})

    return messages

print(build_messages(rand_dev.iloc[0])[-1]["content"])

Classify this entry:
Lemma: béruguette
Definition: Variété d’olive allongée à pulpe ferme.


In [19]:
# Load model and tokenizer for run_icl()
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = shared.MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model and tokenizer loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model and tokenizer loaded successfully!


In [20]:
# Define the ICL evaluation function
def run_icl(
    rows,
    prompt_language="en",
    use_lemma=True,
    few_shot=False,
    batch_size=8,
):
    messages_list = [
        build_messages(
            row,
            prompt_language=prompt_language,
            use_lemma=use_lemma,
            few_shot=few_shot,
        )
        for _, row in rows.iterrows()
    ]

    raw_outputs = shared.generate_batch(
        model,
        tokenizer,
        messages_list,
        batch_size=batch_size,
    )

    metrics, results = shared.evaluate(
        rows,
        raw_outputs,
        hypersense_labels,
        verbose=False,
    )

    return metrics, results

In [21]:
# Test on four examples
test_metrics, test_results = run_icl(
    rand_dev.head(4),
    prompt_language="en",
    use_lemma=True,
    few_shot=False,
    batch_size=4,
)

print(test_metrics)
display(test_results[["lemma", "gold", "raw", "pred", "status"]])

[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'accuracy': 0.5, 'macro_f1': 0.08333333333333333, 'invalid_rate': np.float64(0.25)}


,lemma,gold,raw,pred,status
0,béruguette,inanimate,inanimate,inanimate,valid
1,Reguinois,animate,inhabitant,INVALID,none
2,Saint-Hippolytaise,animate,institution,institution,valid
3,butan-1-ol,inanimate,inanimate,inanimate,valid


In [22]:
# Define the eight experimental conditions
conditions = {
    "E1": {"prompt_language": "en", "use_lemma": True,  "few_shot": False},
    "E2": {"prompt_language": "en", "use_lemma": False, "few_shot": False},
    "E3": {"prompt_language": "en", "use_lemma": True,  "few_shot": True},
    "E4": {"prompt_language": "en", "use_lemma": False, "few_shot": True},
    "E5": {"prompt_language": "fr", "use_lemma": True,  "few_shot": False},
    "E6": {"prompt_language": "fr", "use_lemma": False, "few_shot": False},
    "E7": {"prompt_language": "fr", "use_lemma": True,  "few_shot": True},
    "E8": {"prompt_language": "fr", "use_lemma": False, "few_shot": True},
}

In [23]:
# Run all conditions on rand-dev
OUT_DIR = "/content/icl_results"
os.makedirs(OUT_DIR, exist_ok=True)

dev_metrics = {}
dev_results = {}

for condition_name, config in conditions.items():
    print(f"\nRunning {condition_name}: {config}")

    metrics, results = run_icl(
        rand_dev,
        prompt_language=config["prompt_language"],
        use_lemma=config["use_lemma"],
        few_shot=config["few_shot"],
        batch_size=8,
    )

    dev_metrics[condition_name] = metrics
    dev_results[condition_name] = results

    shared.save_results(
        run_name=condition_name,
        split_name="rand-dev",
        metrics=metrics,
        results=results,
        out_dir=OUT_DIR,
        config=config,
    )

    print(
        f"Accuracy={metrics['accuracy']:.4f}, "
        f"Macro-F1={metrics['macro_f1']:.4f}, "
        f"Invalid rate={metrics['invalid_rate']:.4f}"
    )

[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Running E1: {'prompt_language': 'en', 'use_lemma': True, 'few_shot': False}


[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Saved predictions to: /content/icl_results/E1_rand-dev_preds.csv
Saved metrics to: /content/icl_results/results_summary.csv
Accuracy=0.2333, Macro-F1=0.1573, Invalid rate=0.0912

Running E2: {'prompt_language': 'en', 'use_lemma': False, 'few_shot': False}


[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

...

Saved predictions to: /content/icl_results/E6_rand-dev_preds.csv
Saved metrics to: /content/icl_results/results_summary.csv
Accuracy=0.1667, Macro-F1=0.1887, Invalid rate=0.1246

Running E7: {'prompt_language': 'fr', 'use_lemma': True, 'few_shot': True}


[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Saved predictions to: /content/icl_results/E7_rand-dev_preds.csv
Saved metrics to: /content/icl_results/results_summary.csv
Accuracy=0.4982, Macro-F1=0.3083, Invalid rate=0.0246

Running E8: {'prompt_language': 'fr', 'use_lemma': False, 'few_shot': True}


[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Saved predictions to: /content/icl_results/E8_rand-dev_preds.csv
Saved metrics to: /content/icl_results/results_summary.csv
Accuracy=0.5158, Macro-F1=0.2912, Invalid rate=0.0211


In [24]:
# Compare conditions on rand-dev
dev_summary = pd.DataFrame([
    {
        "condition": name,
        **metrics,
        **conditions[name],
    }
    for name, metrics in dev_metrics.items()
])

dev_summary = dev_summary.sort_values(
    "macro_f1",
    ascending=False,
).reset_index(drop=True)

display(dev_summary)

best_condition = dev_summary.iloc[0]["condition"]
best_config = conditions[best_condition]

print("Best condition:", best_condition)
print("Best configuration:", best_config)

,condition,accuracy,macro_f1,invalid_rate,prompt_language,use_lemma,few_shot
0,E7,0.498246,0.308312,0.024561,fr,True,True
1,E8,0.515789,0.291161,0.021053,fr,False,True
2,E3,0.500000,0.282735,0.043860,en,True,True
3,E4,0.508772,0.275409,0.021053,en,False,True
4,E2,0.254386,0.189731,0.080702,en,False,False
5,E6,0.166667,0.188691,0.124561,fr,False,False
6,E1,0.233333,0.157286,0.091228,en,True,False
7,E5,0.157895,0.129714,0.149123,fr,True,False


Best condition: E7
Best configuration: {'prompt_language': 'fr', 'use_lemma': True, 'few_shot': True}


In [25]:
# Evaluate the best condition on the test sets
test_set_metrics = {}
test_set_results = {}

for split_name, split_data in [
    ("rand-test", rand_test),
    ("freq-test", freq_test),
]:
    print(f"\nEvaluating on {split_name}")

    metrics, results = run_icl(
        split_data,
        prompt_language=best_config["prompt_language"],
        use_lemma=best_config["use_lemma"],
        few_shot=best_config["few_shot"],
        batch_size=8,
    )

    test_set_metrics[split_name] = metrics
    test_set_results[split_name] = results

    shared.save_results(
        run_name=best_condition,
        split_name=split_name,
        metrics=metrics,
        results=results,
        out_dir=OUT_DIR,
        config=best_config,
    )

    print(
        f"Accuracy={metrics['accuracy']:.4f}, "
        f"Macro-F1={metrics['macro_f1']:.4f}, "
        f"Invalid rate={metrics['invalid_rate']:.4f}"
    )


Evaluating on rand-test


[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Saved predictions to: /content/icl_results/E7_rand-test_preds.csv
Saved metrics to: /content/icl_results/results_summary.csv
Accuracy=0.5192, Macro-F1=0.3672, Invalid rate=0.0221

Evaluating on freq-test


[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Saved predictions to: /content/icl_results/E7_freq-test_preds.csv
Saved metrics to: /content/icl_results/results_summary.csv
Accuracy=0.3283, Macro-F1=0.2855, Invalid rate=0.0369


In [26]:
# Summarize all results
final_summary = pd.DataFrame([
    {
        "condition": name,
        "split": "rand-dev",
        **metrics,
    }
    for name, metrics in dev_metrics.items()
] + [
    {
        "condition": best_condition,
        "split": split_name,
        **metrics,
    }
    for split_name, metrics in test_set_metrics.items()
])

display(final_summary)

print("Results saved in:", OUT_DIR)
print(os.listdir(OUT_DIR))

,condition,split,accuracy,macro_f1,invalid_rate
0,E1,rand-dev,0.233333,0.157286,0.091228
1,E2,rand-dev,0.254386,0.189731,0.080702
2,E3,rand-dev,0.500000,0.282735,0.043860
3,E4,rand-dev,0.508772,0.275409,0.021053
4,E5,rand-dev,0.157895,0.129714,0.149123
5,E6,rand-dev,0.166667,0.188691,0.124561
6,E7,rand-dev,0.498246,0.308312,0.024561
7,E8,rand-dev,0.515789,0.291161,0.021053
8,E7,rand-test,0.519174,0.367175,0.022124
9,E7,freq-test,0.328279,0.285530,0.036915


Results saved in: /content/icl_results
['E6_rand-dev_preds.csv', 'E4_rand-dev_preds.csv', 'E7_freq-test_preds.csv', 'E5_rand-dev_preds.csv', 'E1_rand-dev_preds.csv', 'E2_rand-dev_preds.csv', 'E7_rand-dev_preds.csv', 'E7_rand-test_preds.csv', 'E3_rand-dev_preds.csv', 'results_summary.csv', 'E8_rand-dev_preds.csv']


In [27]:
# Error analysis for E7 on both test sets
from sklearn.metrics import precision_recall_fscore_support

for set_name, error_df in test_set_results.items():
    print(f"\n{'=' * 20} {set_name} {'=' * 20}")

    labels = sorted(error_df["gold"].dropna().unique())

    precision, recall, f1, support = precision_recall_fscore_support(
        error_df["gold"],
        error_df["pred"],
        labels=labels,
        zero_division=0
    )

    class_results = pd.DataFrame({
        "hypersense": labels,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support
    }).sort_values("f1", ascending=True)

    print("\nPer-class performance:")
    display(class_results.round(3))

    errors = error_df[error_df["gold"] != error_df["pred"]]

    error_pairs = (
        errors.groupby(["gold", "pred"])
        .size()
        .sort_values(ascending=False)
        .reset_index(name="count")
    )

    print("\nTop 10 errors:")
    display(error_pairs.head(10))


==================== rand-test ====================

Per-class performance:


,hypersense,precision,recall,f1,support
2,dynamic_situation*information,0.000,0.000,0.000,3
7,other,0.048,0.154,0.073,26
8,possession,0.059,0.333,0.100,3
5,information,0.455,0.111,0.179,45
6,institution,0.200,0.429,0.273,7
10,stative_situation,0.611,0.234,0.338,47
4,inanimate*information,0.270,0.588,0.370,17
1,dynamic_situation,0.529,0.351,0.422,77
9,quantity,0.450,0.643,0.529,14
3,inanimate,0.575,0.734,0.645,214



Top 10 errors:


,gold,pred,count
0,animate,inanimate,59
1,inanimate,other,30
2,animate,other,19
3,information,inanimate,17
4,other,inanimate,14
5,stative_situation,dynamic_situation,11
6,information,other,11
7,dynamic_situation,inanimate,10
8,stative_situation,inanimate,9
9,stative_situation,other,8



==================== freq-test ====================

Per-class performance:


,hypersense,precision,recall,f1,support
7,other,0.064,0.123,0.084,163
2,dynamic_situation*information,0.143,0.087,0.108,46
0,animate,0.889,0.063,0.118,127
8,possession,0.083,0.409,0.138,22
5,information,0.339,0.123,0.180,163
4,inanimate*information,0.172,0.270,0.211,37
10,stative_situation,0.452,0.168,0.245,196
6,institution,0.202,0.714,0.315,28
9,quantity,0.426,0.500,0.460,46
3,inanimate,0.471,0.562,0.512,317



Top 10 errors:


,gold,pred,count
0,inanimate,other,92
1,dynamic_situation,other,72
2,animate,inanimate,49
3,information,inanimate,40
4,other,dynamic_situation,40
5,information,other,39
6,stative_situation,inanimate,33
7,other,institution,29
8,stative_situation,other,29
9,animate,other,28


In [28]:
# Compare animate errors across test sets

for set_name, test_df in test_set_results.items():
    animate_df = test_df[test_df["gold"] == "animate"]

    print(f"\n{'=' * 20} {set_name} {'=' * 20}")
    print("Total animate examples:", len(animate_df))

    print("\nPrediction distribution:")
    display(
        animate_df["pred"]
        .value_counts()
        .rename_axis("predicted_label")
        .reset_index(name="count")
    )

    animate_errors = animate_df[animate_df["gold"] != animate_df["pred"]]

    print("\nAnimate error examples:")
    display(
        animate_errors[["lemma", "definition", "gold", "pred"]]
        .head(20)
    )


==================== rand-test ====================
Total animate examples: 218

Prediction distribution:


,predicted_label,count
0,animate,120
1,inanimate,59
2,other,19
3,possession,7
4,institution,4
5,INVALID,3
6,inanimate*information,2
7,dynamic_situation,2
8,time,1
9,dynamic_situation*information,1



Animate error examples:


,lemma,definition,gold,pred
20,damalisque_à_front_blanc,Synonyme de blesbok (antilope).,animate,inanimate
22,Albertaceratops,Type de dinosaure.,animate,inanimate
40,lascar,Individu (avec une connotation plutôt péjorati...,animate,other
41,Vtesse,Vicomtesse.,animate,inanimate
45,iscariotiste,Membre d’une secte qui vénérait Judas Iscariot...,animate,other
46,spiroptère,Genre d’helminthes à corps cylindrique aminci ...,animate,inanimate
51,facilitrice,"Celle qui facilite, rend aisée, une action.",animate,possession
64,homme_du_rang,Simple soldat.,animate,inanimate
65,clam,Nom vernaculaire de plusieurs espèces de mollu...,animate,inanimate
66,frère_trois-points,Personne membre de la franc-maçonnerie.,animate,institution



==================== freq-test ====================
Total animate examples: 127

Prediction distribution:


,predicted_label,count
0,inanimate,49
1,other,28
2,institution,16
3,possession,12
4,INVALID,9
5,animate,8
6,dynamic_situation,2
7,stative_situation,2
8,information,1



Animate error examples:


,lemma,definition,gold,pred
18,mission,Les missionnaires eux-mêmes.,animate,INVALID
21,rang,Ensemble de soldats alignés les uns à côté des...,animate,inanimate
27,faction,Groupement dans une assemblée.,animate,INVALID
28,milieu,"Pègre, l’ensemble des délinquants et des crimi...",animate,other
36,société,Petit groupe de personnes entretenant des lien...,animate,inanimate
37,société,Petit groupe de personnes réuni autour de quel...,animate,other
41,division,Groupe d’élèves suivant les enseignements ense...,animate,other
54,cabinet,Ensemble des collaborateurs d’un ministre.,animate,institution
55,cour,"Entourage d’une personne, gens empressés à lui...",animate,other
56,cour,Le souverain et ses ministres ; le gouvernemen...,animate,institution
